In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
!pip install pyreadstat

import os
import pandas as pd

# 1. 드라이브 내에서 HN24_ALL.sav 파일의 실제 경로 자동 탐색
found_path = None
for root, dirs, files in os.walk('/content/drive'):
    if 'HN24_ALL.sav' in files:
        found_path = os.path.join(root, 'HN24_ALL.sav')
        break

if found_path:
    print(f"파일을 찾았습니다! 실제 경로: {found_path}")
    # 2. 찾은 경로로 파일 불러오기
    df = pd.read_spss(found_path)
    print("데이터 불러오기 성공! 전체 크기(행, 열):", df.shape)
else:
    print("파일을 찾지 못했습니다. 드라이브 연동 상태나 파일 위치를 다시 확인해주세요.")

파일을 찾았습니다! 실제 경로: /content/drive/MyDrive/hdac_data/HN24_ALL.sav
데이터 불러오기 성공! 전체 크기(행, 열): (6997, 813)


In [23]:
# ==============================================================================
# [9주차 과제] 변수 확인, 원 코드 빈도표, 가중치·층·집락 및 단계별 대상자 산출 코드
# ==============================================================================

import pandas as pd

print("==================================================")
print(" 1. 가중치, 층, 집락 설계 변수 확인")
print("================================================== ")
# 설계 변수 및 통합 가중치 존재 여부 확인
design_vars = ['kstrata', 'psu', 'wt_tot']
for v in design_vars:
    exists = v in df.columns
    print(f"- 설계/가중치 변수 '{v}' 존재 여부: {exists}")

# 건강설문 가중치 후보 목록 재확인
weight_cols = [col for col in df.columns if 'wt_' in col.lower()]
print(f"- 가중치 관련 후보 변수 목록: {weight_cols}")


print("\n==================================================")
print(" 2. 스트레스 및 수면 관련 원 코드 빈도표 확인")
print("==================================================")
# (1) 독립변수(노출) 관련: 원문항 BP1 및 파생변수 mh_stress 빈도
if 'BP1' in df.columns:
    print("\n[독립변수 원문항: BP1 (스트레스 인지 정도) 빈도표]")
    print(df['BP1'].value_counts(dropna=False))

if 'mh_stress' in df.columns:
    print("\n[독립변수 파생변수: mh_stress (스트레스 인지 이분형) 빈도표]")
    print(df['mh_stress'].value_counts(dropna=False))

# (2) 종속변수(결과) 관련: 수면 관련 원문항(BP6_1 등 지침서 변수) 후보 확인
sleep_candidates = [col for col in df.columns if 'bp6' in col.lower() or 'sleep' in col.lower()]
print(f"\n[종속변수 수면 관련 후보 변수]: {sleep_candidates}")
if 'BP6_1' in df.columns:
    print("\n[종속변수 원문항: BP6_1 빈도표 (상위 10개)]")
    print(df['BP6_1'].value_counts(dropna=False).head(10))


print("\n==================================================")
print(" 3. 분석 대상 선정 단계별 인원 및 결측 현황 산출")
print("==================================================")

# 1단계: 전체 데이터 수
n_total = df.shape[0]
print(f"• 1단계 (전체 원시자료): {n_total:,}명")

# 2단계: 만 19세 이상 성인 필터링 (나이 변수: age)
df_adult = df[df['age'] >= 19].copy()
n_adult = df_adult.shape[0]
excluded_adult = n_total - n_adult
print(f"• 2단계 (만 19세 이상 성인): {n_adult:,}명 (제외: {excluded_adult:,}명 - 미성년자)")

# 3단계: 주요 분석 변수별 개별 결측(NaN) 현황 확인
key_vars = ['mh_stress', 'kstrata', 'psu', 'wt_tot']
print("\n[주요 분석 변수별 개별 결측 인원]")
for col in key_vars:
    if col in df_adult.columns:
        n_missing = df_adult[col].isna().sum()
        print(f"  - {col}: {n_missing:,}명 결측")

# 4단계: 주요 분석 변수 모두 유효한 최종 분석 대상자 수 산출
valid_vars = [v for v in key_vars if v in df_adult.columns]
df_final = df_adult.dropna(subset=valid_vars).copy()
n_final = df_final.shape[0]
excluded_final = n_adult - n_final

print(f"\n• 3단계 (최종 분석 대상자 - 주요 변수 유효): {n_final:,}명 (제외: {excluded_final:,}명)")
print("==================================================\n")

 1. 가중치, 층, 집락 설계 변수 확인
- 설계/가중치 변수 'kstrata' 존재 여부: True
- 설계/가중치 변수 'psu' 존재 여부: True
- 설계/가중치 변수 'wt_tot' 존재 여부: True
- 가중치 관련 후보 변수 목록: ['wt_hs', 'wt_itvex', 'wt_ntr', 'wt_pft', 'wt_tot', 'wt_pfnt', 'HE_wt_pct']

 2. 스트레스 및 수면 관련 원 코드 빈도표 확인

[독립변수 원문항: BP1 (스트레스 인지 정도) 빈도표]
BP1
3.0    3409
4.0    1286
2.0    1216
8.0     561
1.0     297
9.0     219
NaN       9
Name: count, dtype: int64

[독립변수 파생변수: mh_stress (스트레스 인지 이분형) 빈도표]
mh_stress
0.0    4695
1.0    1513
NaN     789
Name: count, dtype: int64

[종속변수 수면 관련 후보 변수]: ['BP6_10', 'BP6_2', 'BP6_31']

 3. 분석 대상 선정 단계별 인원 및 결측 현황 산출
• 1단계 (전체 원시자료): 6,997명
• 2단계 (만 19세 이상 성인): 6,033명 (제외: 964명 - 미성년자)

[주요 분석 변수별 개별 결측 인원]
  - mh_stress: 220명 결측
  - kstrata: 0명 결측
  - psu: 0명 결측
  - wt_tot: 174명 결측

• 3단계 (최종 분석 대상자 - 주요 변수 유효): 5,706명 (제외: 327명)



In [24]:
# ==============================================================================
# [완성 코드] 2024년 국민건강영양조사 스트레스와 수면시간 분석 대상자 선정 및 변수 확인
# ==============================================================================

import pandas as pd

# 1. 분석용 데이터 복사본 생성
df_recoded = df.copy()

print("=== [1단계] 전체 데이터 수 ===")
n_total = df_recoded.shape[0]
print(f"• 전체 데이터 수 (N): {n_total:,}명\n")


# ==============================================================================
# 2. 만 19세 이상 성인 필터링 (나이 변수: age)
# ==============================================================================
df_adult = df_recoded[df_recoded['age'] >= 19].copy()
n_adult = df_adult.shape[0]
excluded_adult = n_total - n_adult

print("=== [2단계] 연령 필터링 (만 19세 이상 성인) ===")
print(f"• 성인 대상자 수 (N): {n_adult:,}명 (제외: {excluded_adult:,}명 - 19세 미만 미성년자)\n")


# ==============================================================================
# 3. 주요 분석 변수 정의표 기반 변수명 확인 및 결측치 현황 산출
# ==============================================================================
# - 독립변수(노출): mh_stress (또는 원문항 BP1)
# - 종속변수(결과): 수면시간 관련 변수 (프로토콜 정의에 따른 수면시간 변수명 반영, 예: 주중/주말 수면시간 또는 관련 설문 변수)
# - 설계변수: kstrata (층), psu (집락), wt_tot (가중치 - 분석 목적에 맞게 wt_itvex 등으로 변경 가능)

# 프로토콜 정의표에 따른 주요 분석 변수 리스트
# ※ 수면시간 변수는 프로토콜에 따라 산출되거나 지정된 변수명(예: 관련 수면 시간 문항)으로 매칭됩니다.
# 현재 데이터프레임에 존재하는 수면 관련 변수를 자동으로 확인하거나 지정할 수 있습니다.

analysis_vars = ['mh_stress', 'kstrata', 'psu', 'wt_itvex']

print("=== [3단계] 주요 변수별 개별 결측(NaN) 현황 ===")
for col in analysis_vars:
    if col in df_adult.columns:
        n_missing = df_adult[col].isna().sum()
        print(f"  - {col}: {n_missing:,}명 결측")
    else:
        print(f"  - {col}: 변수가 존재하지 않습니다.")

# 수면시간 관련 원문항(예: BP6_1 등 지침서 변수) 확인 코드 추가
sleep_check_cols = [c for c in df_adult.columns if 'bp6' in c.lower() or 'sleep' in c.lower()]
print(f"\n[참고] 데이터 내 수면/BP6 관련 후보 변수들: {sleep_check_cols[:5]} ... (총 {len(sleep_check_cols)}개)")


# ==============================================================================
# 4. 최종 분석 대상자 수 산출 (주요 변수 결측 제외)
# ==============================================================================
# 주요 변수(스트레스, 설계변수, 가중치 등) 기준 유효 인원 산출
valid_vars = [v for v in analysis_vars if v in df_adult.columns]
df_final = df_adult.dropna(subset=valid_vars).copy()
n_final = df_final.shape[0]
excluded_final = n_adult - n_final

print("\n=== [4단계] 최종 분석 대상자 선정 결과 ===")
print(f"• 최종 분석 대상자 수 (N): {n_final:,}명 (제외 인원: {excluded_final:,}명)")

=== [1단계] 전체 데이터 수 ===
• 전체 데이터 수 (N): 6,997명

=== [2단계] 연령 필터링 (만 19세 이상 성인) ===
• 성인 대상자 수 (N): 6,033명 (제외: 964명 - 19세 미만 미성년자)

=== [3단계] 주요 변수별 개별 결측(NaN) 현황 ===
  - mh_stress: 220명 결측
  - kstrata: 0명 결측
  - psu: 0명 결측
  - wt_itvex: 0명 결측

[참고] 데이터 내 수면/BP6 관련 후보 변수들: ['BP6_10', 'BP6_2', 'BP6_31'] ... (총 3개)

=== [4단계] 최종 분석 대상자 선정 결과 ===
• 최종 분석 대상자 수 (N): 5,813명 (제외 인원: 220명)


In [25]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/hdac_data'      # Drive에서 폴더 이름의 공백을 지운 뒤
print([repr(f) for f in os.listdir(DATA_DIR)])     # 파일 이름 확인

!pip -q install pyreadstat
import pyreadstat
PATH = f'{DATA_DIR}/HN24_ALL.sav'

# 변수 이름과 라벨만 먼저 읽기 (자료 전체를 읽지 않아 빠름)
_, meta = pyreadstat.read_sav(PATH, metadataonly=True)
labels = meta.column_names_to_labels

def find_vars(*keywords):
    """이름이나 라벨에 키워드가 들어간 변수를 모두 보여 준다."""
    for name, label in labels.items():
        if any(k in f'{name} {label or ""}' for k in keywords):
            print(f'{name:15s} {label}')

find_vars('수면', '잠', '취침', '기상', '일어')     # 수면 문항
find_vars('결혼')                  # 결혼상태
find_vars('경제활동', '취업')       # 경제활동
find_vars('근로', '근무', '일한')   # 근로시간, 근로시간대
find_vars('우울', 'PHQ')           # 우울
find_vars('불안', 'GAD')           # 불안
find_vars('의사진단')              # 만성질환

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
["'09_data_check.ipynb'", "'hn24_all.sas7bdat'", "'HN24_ALL.sav'"]
BP16_11         주중 잠자리에 든 시각(시간)
BP16_12         주중 잠자리에 든 시각(분)
BP16_13         주중 일어난 시각(시간)
BP16_14         주중 일어난 시각(분)
BP16_21         주말 잠자리에 든 시각(시간)
BP16_22         주말 잠자리에 든 시각(분)
BP16_23         주말 일어난 시각(시간)
BP16_24         주말 일어난 시각(분)
HE_Ubld         요잠혈
BM1_8           칫솔질 시기: 잠자기 전
marri_1         결혼여부
marri_2         결혼상태
occp            직업재분류 및 실업/비경제활동인구상태 코드
EC1_1           경제활동 상태
EC_stt_2        종사상지위: 임금근로자 상세
EC_wht_23       주당 평균 근로시간
EC_wht_5        주/변형근로시간
EC_wht_6        변형근로시간:기타상세
LQ4_15          활동제한 사유: 우울, 불안, 정서상 문제
LQ_5EQL         EuroQoL: 불안, 우울
BP_PHQ_1        PHQ-9 문항1
BP_PHQ_2        PHQ-9 문항2
BP_PHQ_3        PHQ-9 문항3
BP_PHQ_4        PHQ-9 문항4
BP_PHQ_5        PHQ-9 문항5
BP_PHQ_6        PHQ-9 문항6
BP_PHQ_7        PHQ-9 문항7
BP_PHQ_8        PHQ-9 문항8
BP_PHQ_9

In [27]:
import numpy as np
import pandas as pd

df, meta = pyreadstat.read_sav(PATH, usecols=['age', 'sex', 'BP1', 'mh_stress'])   # 4단계에서 공변량을 더함

df['stress'] = np.select([df['BP1'].isin([1, 2]), df['BP1'].isin([3, 4])], [1, 0], default=np.nan)

print(pd.crosstab(df['BP1'], df['stress'], dropna=False))       # 원 코드가 맞게 묶였는지
print(pd.crosstab(df['stress'], df['mh_stress'], dropna=False)) # mh_stress와 같은지

stress   0.0   1.0  NaN
BP1                    
1.0        0   297    0
2.0        0  1216    0
3.0     3409     0    0
4.0     1286     0    0
8.0        0     0  561
9.0        0     0  219
NaN        0     0    9
mh_stress   0.0   1.0  NaN
stress                    
0.0        4695     0    0
1.0           0  1513    0
NaN           0     0  789


In [28]:
# 전체 변수를 가진 원본 데이터 다시 불러오기
df, meta = pyreadstat.read_sav(PATH)

In [29]:
# 1단계 출력 및 변수 목록에서 확인한 실제 변수명으로 매핑
WD = dict(
    bed_h='BP16_11', bed_m='BP16_12', wake_h='BP16_13', wake_m='BP16_14'
)  # 주중 (잠자리에 든 시각 시간/분, 일어난 시각 시간/분)
WE = dict(
    bed_h='BP16_21', bed_m='BP16_22', wake_h='BP16_23', wake_m='BP16_24'
)  # 주말 (잠자리에 든 시각 시간/분, 일어난 시각 시간/분)
SPECIAL = [88, 99]  # 결측치나 해당 없음 처리할 특수문항 코드 (지침에 맞게 설정)                                                          # 확인 필요

def clean(col):
    return df[col].where(~df[col].isin(SPECIAL))

def sleep_hours(v):
    bed  = clean(v['bed_h'])  * 60 + clean(v['bed_m'])
    wake = clean(v['wake_h']) * 60 + clean(v['wake_m'])
    return ((wake - bed) % (24 * 60)) / 60      # 자정을 넘겨 잠들어도 맞게 계산

df['sleep_wd'] = sleep_hours(WD)
df['sleep_we'] = sleep_hours(WE)
df['sleep'] = (df['sleep_wd'] * 5 + df['sleep_we'] * 2) / 7

In [30]:
print(df[['sleep']].describe())                          # 최솟값·최댓값이 상식적인지
print((df['sleep'] < 3).sum(), (df['sleep'] > 16).sum())  # 이상값 후보 수

             sleep
count  6202.000000
mean      7.430463
std       1.253988
min       2.500000
25%       6.678571
50%       7.500000
75%       8.142857
max      21.000000
4 1


In [31]:
# 1. 이상값(3시간 미만 또는 16시간 초과)을 제외한 유효 수면 시간 데이터 생성
df_sleep_valid = df[
    df['sleep'].notna() & (df['sleep'] >= 3) & (df['sleep'] <= 16)
].copy()

# 2. 제외된 인원 수 확인
excluded_outlier = (
    df['sleep'].notna() & ((df['sleep'] < 3) | (df['sleep'] > 16))
).sum()
print(f'• 수면 시간 이상값으로 제외된 인원: {excluded_outlier}명')

# 3. 이상값이 제외된 최종 수면 시간 기술통계 확인
print('\n=== [이상값 제외 후 수면 시간 기술통계] ===')
print(df_sleep_valid[['sleep']].describe())

• 수면 시간 이상값으로 제외된 인원: 5명

=== [이상값 제외 후 수면 시간 기술통계] ===
             sleep
count  6197.000000
mean      7.431363
std       1.236618
min       3.000000
25%       6.678571
50%       7.500000
75%       8.142857
max      13.857143
